# AkidaNet on CIFAR-10: train, quantize, convert to Akida

Pipeline:
1. Load CIFAR-10, resize on the fly.
2. Build **AkidaNet** (ImageNet-pretrained, from `akida_models`) + small classifier head.
3. Train head, then fine-tune whole net (float).
4. **Quantize** with `quantizeml` (needed: Akida 1.0 runs integer weights/activations only), then QAT fine-tune to recover accuracy.
5. **Convert** with `cnn2snn` to an Akida model, check accuracy, save `.fbz`.

Needs `akida`, `cnn2snn`, `quantizeml`, `akida_models` (MetaTF; see `requirements-akida.txt`). Checked against MetaTF 2.19 docs (doc.brainchipinc.com). Still **not executed**. Not confirmed by docs: QAT via plain Keras `fit` on the quantized model, and Dense-after-GAP convertibility (`check_model_compatibility` covers the latter). Versions are printed in the first cell.

In [ ]:
import os
os.environ["TF_USE_LEGACY_KERAS"] = "1"   # MetaTF tools expect Keras 2

import numpy as np
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers

import akida, cnn2snn, quantizeml, akida_models
from akida_models import akidanet_imagenet, akidanet_imagenet_pretrained
from quantizeml.models import quantize, QuantizationParams
from cnn2snn import convert, check_model_compatibility
try:
    from cnn2snn import set_akida_version, AkidaVersion
except ImportError:
    from akida import AkidaVersion
    from cnn2snn import set_akida_version

# cnn2snn targets Akida 2.0 by default. 4-bit weights/activations and the AKD1000 board are Akida 1.0,
# so build, quantize and convert all run inside `with set_akida_version(AkidaVersion.v1):` (as in the MetaTF AkidaNet example).

for m in (tf, akida, cnn2snn, quantizeml, akida_models):
    print(m.__name__, getattr(m, "__version__", "?"))

In [ ]:
# ---- config ----
IMG_SIZE      = 128     # AkidaNet is fully convolutional; 128 is much faster than 224 (pretrained weights still load)
ALPHA         = 0.5     # AkidaNet width multiplier (pretrained weights exist for 0.25 / 0.5 / 1.0)
BATCH         = 64
EPOCHS_HEAD   = 5       # float: base frozen
EPOCHS_FINE   = 10      # float: everything trainable
EPOCHS_QAT    = 5       # quantization-aware fine-tune
WEIGHT_BITS   = 4       # 4 or 8; Akida 1.0: 4-bit weights/activations, 8-bit first-layer weights
ACT_BITS      = 4       # 1/2/4
N_CALIB       = 1024    # calibration images for quantization
SMOKE_TEST    = False   # True = tiny subset, 1 epoch each, to check the pipeline end-to-end

NUM_CLASSES = 10

## 1. Data

In [ ]:
(x_train, y_train), (x_test, y_test) = keras.datasets.cifar10.load_data()
y_train, y_test = y_train.ravel(), y_test.ravel()
if SMOKE_TEST:
    x_train, y_train = x_train[:2000], y_train[:2000]
    x_test,  y_test  = x_test[:500],   y_test[:500]
    EPOCHS_HEAD = EPOCHS_FINE = EPOCHS_QAT = 1
print(x_train.shape, x_test.shape)

def prep_train(img, label):
    img = tf.cast(img, tf.float32)
    img = tf.image.random_flip_left_right(img)
    img = tf.image.resize_with_crop_or_pad(img, 36, 36)
    img = tf.image.random_crop(img, (32, 32, 3))
    img = tf.image.resize(img, (IMG_SIZE, IMG_SIZE))
    return img, label                      # raw 0-255, scaling happens inside the model

def prep_test(img, label):
    img = tf.image.resize(tf.cast(img, tf.float32), (IMG_SIZE, IMG_SIZE))
    return img, label

def make_ds(x, y, train):
    ds = tf.data.Dataset.from_tensor_slices((x, y))
    if train:
        ds = ds.shuffle(10000)
    ds = ds.map(prep_train if train else prep_test, num_parallel_calls=tf.data.AUTOTUNE)
    return ds.batch(BATCH).prefetch(tf.data.AUTOTUNE)

train_ds = make_ds(x_train, y_train, True)
test_ds  = make_ds(x_test,  y_test,  False)

## 2. Model: pretrained AkidaNet + head

Head uses only Akida-convertible layers (GAP, Dense, Dropout). No softmax: model outputs logits, loss uses `from_logits=True`.

In [ ]:
with set_akida_version(AkidaVersion.v1):
    base = akidanet_imagenet(input_shape=(IMG_SIZE, IMG_SIZE, 3), alpha=ALPHA,
                             include_top=False, pooling=None)

    # Float ImageNet weights from the model zoo (documented: akidanet_imagenet_pretrained(alpha, quantized=...)).
    # Conv weights do not depend on input resolution, so copy layer by layer by name (224 -> IMG_SIZE).
    try:
        pre = akidanet_imagenet_pretrained(ALPHA, quantized=False)
        copied = 0
        for l in base.layers:
            if l.weights:
                try:
                    l.set_weights(pre.get_layer(l.name).get_weights()); copied += 1
                except (ValueError, KeyError):
                    pass
        print(f"copied pretrained weights into {copied} layers")
    except Exception as e:
        print("WARNING: pretrained weights not loaded, training from scratch ->", repr(e))

print([type(l).__name__ for l in base.layers[:4]])

In [ ]:
# Make sure the model takes raw 0-255 input (Akida needs uint8 input). Recent akida_models builds
# a Rescaling layer into AkidaNet; add one only if it is missing.
inputs = keras.Input((IMG_SIZE, IMG_SIZE, 3))
has_rescaling = any(isinstance(l, layers.Rescaling) for l in base.layers[:3])
x = inputs if has_rescaling else layers.Rescaling(1/127.5, offset=-1)(inputs)
x = base(x)
x = layers.GlobalAveragePooling2D()(x)
x = layers.Dropout(0.3)(x)
outputs = layers.Dense(NUM_CLASSES)(x)
model = keras.Model(inputs, outputs, name="akidanet_cifar10")
model.summary(line_length=100, expand_nested=False)

## 3. Float training

In [ ]:
def compile_float(lr):
    model.compile(optimizer=keras.optimizers.Adam(lr),
                  loss=keras.losses.SparseCategoricalCrossentropy(from_logits=True),
                  metrics=["accuracy"])

# 3a. head only
base.trainable = False
compile_float(1e-3)
model.fit(train_ds, validation_data=test_ds, epochs=EPOCHS_HEAD)

# 3b. fine-tune everything, cosine LR decay
base.trainable = True
steps = EPOCHS_FINE * len(train_ds)
compile_float(keras.optimizers.schedules.CosineDecay(1e-4, steps))
model.fit(train_ds, validation_data=test_ds, epochs=EPOCHS_FINE)

float_loss, float_acc = model.evaluate(test_ds)
print(f"float accuracy: {float_acc:.4f}")
model.save("akidanet_cifar10_float.h5")

## 4. Quantization

Required: Akida 1.0 hardware only executes low-bit integer weights/activations. `quantizeml` does post-training quantization using calibration samples; QAT then fine-tunes the quantized model.

In [ ]:
# calibration samples: raw 0-255 images, same preprocessing as test
calib = np.concatenate([xb.numpy() for xb, _ in train_ds.take(int(np.ceil(N_CALIB / BATCH)))])[:N_CALIB].astype(np.float32)
print("calibration:", calib.shape, calib.min(), calib.max())

qparams = QuantizationParams(input_weight_bits=8, weight_bits=WEIGHT_BITS, activation_bits=ACT_BITS)
with set_akida_version(AkidaVersion.v1):
    qmodel = quantize(model, qparams=qparams, samples=calib, num_samples=len(calib), batch_size=BATCH)

qmodel.compile(optimizer=keras.optimizers.Adam(1e-4),
               loss=keras.losses.SparseCategoricalCrossentropy(from_logits=True),
               metrics=["accuracy"])
_, ptq_acc = qmodel.evaluate(test_ds)
print(f"post-training quantized ({WEIGHT_BITS}w/{ACT_BITS}a) accuracy: {ptq_acc:.4f}")

In [ ]:
# Quantization-aware fine-tune (skip if PTQ accuracy is already within ~1% of float)
qmodel.fit(train_ds, validation_data=test_ds, epochs=EPOCHS_QAT)
_, q_acc = qmodel.evaluate(test_ds)
print(f"after QAT accuracy: {q_acc:.4f}   (float: {float_acc:.4f})")
qmodel.save("akidanet_cifar10_quantized.h5")

## 5. Convert to Akida

In [ ]:
with set_akida_version(AkidaVersion.v1):
    check_model_compatibility(qmodel)     # raises/warns on unsupported layers (e.g. Dense after GAP) before converting
    akida_model = convert(qmodel)
akida_model.summary()
print("input :", akida_model.input_shape)
print("output:", akida_model.output_shape)

In [ ]:
# Compare quantized Keras vs Akida on test images. Akida takes uint8 NHWC.
N_EVAL = 500 if SMOKE_TEST else 2000
xs, ys = [], []
for xb, yb in test_ds.take(int(np.ceil(N_EVAL / BATCH))):
    xs.append(xb.numpy()); ys.append(yb.numpy())
xe = np.concatenate(xs)[:N_EVAL].astype(np.float32)
ye = np.concatenate(ys)[:N_EVAL]

q_pred = qmodel.predict(xe, verbose=0).argmax(axis=1)
a_out  = np.asarray(akida_model.predict(xe.astype(np.uint8)))
a_pred = a_out.reshape(len(xe), -1).argmax(axis=1)

print(f"quantized keras : {np.mean(q_pred == ye):.4f}")
print(f"akida (software): {np.mean(a_pred == ye):.4f}")
print(f"agreement akida vs quantized: {np.mean(a_pred == q_pred):.4f}")

In [ ]:
# Save; optionally map on a real device (AKD1000) if one is present.
akida_model.save("akidanet_cifar10.fbz")

devices = akida.devices()
print("devices:", devices)
if devices:
    akida_model.map(devices[0])
    akida_model.summary()
    hw_out = np.asarray(akida_model.predict(xe[:200].astype(np.uint8)))
    hw_pred = hw_out.reshape(200, -1).argmax(axis=1)
    print(f"akida (hardware) accuracy on 200 imgs: {np.mean(hw_pred == ye[:200]):.4f}")
else:
    print("no Akida device: ran in software simulation only")